## Bin Packing — meta-heurística de solução única

In [2]:
import argparse
import copy
import random
import sys
import time

CAPACIDADE = 1.0


# (a) REPRESENTACAO DA SOLUCAO
# bins[b] = lista de indices dos itens alocados ao container b
# sizes[u] = tamanho s_u do item u (0 <= s_u <= 1)

def carga_bin(bins, sizes, b):
  return sum(sizes[i] for i in bins[b])


def limpar_bins_vazios(bins):
  return [b for b in bins if b]


def copiar_solucao(bins):
  return [list(b) for b in bins]


# (b) FUNCAO DE AVALIACAO
# custo = numero de containers usados (minimizar k)

def avaliar(bins, sizes):
  bins = limpar_bins_vazios(bins)
  if not all(carga_bin(bins, sizes, b) <= CAPACIDADE for b in range(len(bins))):
    return float('inf')
  return len(bins)


def solucao_inicial_ffd(sizes):
  """First Fit Decreasing: solucao inicial viavel."""
  itens = sorted(range(len(sizes)), key=lambda i: sizes[i], reverse=True)
  bins = [[]]

  for item in itens:
    colocado = False
    for b in range(len(bins)):
      if carga_bin(bins, sizes, b) + sizes[item] <= CAPACIDADE:
        bins[b].append(item)
        colocado = True
        break
    if not colocado:
      bins.append([item])

  return limpar_bins_vazios(bins)


# (c) BUSCA LOCAL — vizinhanca por troca/movimentacao entre containers

def vizinhos(bins, sizes):
  bins = limpar_bins_vazios(bins)
  n_bins = len(bins)

  for i in range(n_bins):
    for j in range(i + 1, n_bins):
      carga_i = carga_bin(bins, sizes, i)
      carga_j = carga_bin(bins, sizes, j)

      # mover item de i para j
      for u in bins[i]:
        if carga_j + sizes[u] <= CAPACIDADE:
          vizinho = copiar_solucao(bins)
          vizinho[i].remove(u)
          vizinho[j].append(u)
          yield limpar_bins_vazios(vizinho)

      # mover item de j para i
      for v in bins[j]:
        if carga_i + sizes[v] <= CAPACIDADE:
          vizinho = copiar_solucao(bins)
          vizinho[j].remove(v)
          vizinho[i].append(v)
          yield limpar_bins_vazios(vizinho)

      # trocar item u (em i) por item v (em j)
      for u in bins[i]:
        for v in bins[j]:
          nova_carga_i = carga_i - sizes[u] + sizes[v]
          nova_carga_j = carga_j - sizes[v] + sizes[u]
          if nova_carga_i <= CAPACIDADE and nova_carga_j <= CAPACIDADE:
            vizinho = copiar_solucao(bins)
            vizinho[i].remove(u)
            vizinho[j].remove(v)
            vizinho[i].append(v)
            vizinho[j].append(u)
            yield limpar_bins_vazios(vizinho)


def busca_local_melhor_melhoria(bins, sizes, limite_tempo):
  inicio = time.time()
  atual = limpar_bins_vazios(copiar_solucao(bins))
  custo_atual = avaliar(atual, sizes)
  iteracoes = 0

  while time.time() - inicio < limite_tempo:
    iteracoes += 1
    melhor_vizinho = None
    melhor_custo = custo_atual

    for vizinho in vizinhos(atual, sizes):
      custo = avaliar(vizinho, sizes)
      if custo < melhor_custo:
        melhor_custo = custo
        melhor_vizinho = vizinho

    if melhor_vizinho is None:
      break

    atual = melhor_vizinho
    custo_atual = melhor_custo

  return atual, custo_atual, iteracoes, time.time() - inicio


# (d) CRITERIO DE PARADA — limite de tempo em segundos (argumento de linha de comando)

def parse_args():
  parser = argparse.ArgumentParser(
    description='Bin Packing com busca local (melhor melhoria)'
  )
  parser.add_argument(
    'tempo',
    type=float,
    nargs='?',
    default=5.0,
    help='limite de tempo em segundos'
  )
  parser.add_argument(
    '--tamanhos',
    type=str,
    default=None,
    help='tamanhos dos itens separados por virgula (ex.: 0.4,0.3,0.2)'
  )

  if 'ipykernel' in sys.modules:
    return parser.parse_args([])
  return parser.parse_args()


def resolver_bin_packing(sizes, limite_tempo):
  inicial = solucao_inicial_ffd(sizes)
  solucao, custo, iteracoes, tempo = busca_local_melhor_melhoria(inicial, sizes, limite_tempo)
  return {
    'solucao_inicial': inicial,
    'solucao': solucao,
    'custo_inicial': avaliar(inicial, sizes),
    'custo': custo,
    'iteracoes': iteracoes,
    'tempo': tempo,
  }


def imprimir_resultado(sizes, resultado):
  print('\nPROBLEMA DE BIN PACKING — BUSCA LOCAL\n')
  print(f"Numero de itens: {len(sizes)}")
  print(f"Tamanhos: {[round(s, 4) for s in sizes]}\n")

  print(f"Solucao inicial (FFD): {resultado['custo_inicial']} containers")
  for b, itens in enumerate(resultado['solucao_inicial'], start=1):
    carga = sum(sizes[i] for i in itens)
    print(f"  Container {b}: itens {itens} | carga = {carga:.4f}")

  print(f"\nSolucao final: {resultado['custo']} containers")
  print(f"Iteracoes de busca local: {resultado['iteracoes']}")
  print(f"Tempo de busca local: {resultado['tempo']:.4f} s\n")

  for b, itens in enumerate(resultado['solucao'], start=1):
    carga = sum(sizes[i] for i in itens)
    print(f"  Container {b}: itens {itens} | carga = {carga:.4f}")


if __name__ == '__main__':
  args = parse_args()

  if args.tamanhos:
    sizes = [float(x.strip()) for x in args.tamanhos.split(',')]
  else:
    random.seed(42)
    sizes = [round(random.random(), 2) for _ in range(20)]

  for s in sizes:
    if not (0 <= s <= 1):
      raise ValueError(f'Tamanho invalido: {s}. Deve satisfazer 0 <= s_u <= 1.')

  resultado = resolver_bin_packing(sizes, args.tempo)
  imprimir_resultado(sizes, resultado)


PROBLEMA DE BIN PACKING — BUSCA LOCAL

Numero de itens: 20
Tamanhos: [0.64, 0.03, 0.28, 0.22, 0.74, 0.68, 0.89, 0.09, 0.42, 0.03, 0.22, 0.51, 0.03, 0.2, 0.65, 0.54, 0.22, 0.59, 0.81, 0.01]

Solucao inicial (FFD): 9 containers
  Container 1: itens [6, 7, 19] | carga = 0.9900
  Container 2: itens [18, 1, 9, 12] | carga = 0.9000
  Container 3: itens [4, 3] | carga = 0.9600
  Container 4: itens [5, 2] | carga = 0.9600
  Container 5: itens [14, 10] | carga = 0.8700
  Container 6: itens [0, 16] | carga = 0.8600
  Container 7: itens [17, 13] | carga = 0.7900
  Container 8: itens [15, 8] | carga = 0.9600
  Container 9: itens [11] | carga = 0.5100

Solucao final: 9 containers
Iteracoes de busca local: 1
Tempo de busca local: 0.0004 s

  Container 1: itens [6, 7, 19] | carga = 0.9900
  Container 2: itens [18, 1, 9, 12] | carga = 0.9000
  Container 3: itens [4, 3] | carga = 0.9600
  Container 4: itens [5, 2] | carga = 0.9600
  Container 5: itens [14, 10] | carga = 0.8700
  Container 6: itens [0,